# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [2]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [3]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [4]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [5]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [6]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [7]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

### Step 1: Check the data
`printSchema()` shows that `inferSchema` read PATENT as an integer and POSTATE as a string. There are 2,923,922 patents and 1,784,989 of them have a state; the rest have NULL. No state is stored as an empty string, so NULL is the only "no state" marker. There are 16,522,438 citation pairs.

In [8]:
patents.printSchema()
patents.agg(count("*").alias("patents"), count("POSTATE").alias("with_state")).show()
print("empty-string states:", patents.filter(col("POSTATE") == "").count())
print("citations:", citations.count())

root
 |-- PATENT: integer (nullable = true)
 |-- GYEAR: integer (nullable = true)
 |-- GDATE: integer (nullable = true)
 |-- APPYEAR: integer (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- POSTATE: string (nullable = true)
 |-- ASSIGNEE: integer (nullable = true)
 |-- ASSCODE: integer (nullable = true)
 |-- CLAIMS: integer (nullable = true)
 |-- NCLASS: integer (nullable = true)
 |-- CAT: integer (nullable = true)
 |-- SUBCAT: integer (nullable = true)
 |-- CMADE: integer (nullable = true)
 |-- CRECEIVE: integer (nullable = true)
 |-- RATIOCIT: double (nullable = true)
 |-- GENERAL: double (nullable = true)
 |-- ORIGINAL: double (nullable = true)
 |-- FWDAPLAG: double (nullable = true)
 |-- BCKGTLAG: double (nullable = true)
 |-- SELFCTUB: double (nullable = true)
 |-- SELFCTLB: double (nullable = true)
 |-- SECDUPBD: double (nullable = true)
 |-- SECDLWBD: double (nullable = true)

+-------+----------+
|patents|with_state|
+-------+----------+
|2923922|   1784989|
+----

### Step 2: Put the state of both patents on every citation
Two small lookup tables (patent number to state) are built from `patents`, one with the key renamed to CITING and one renamed to CITED. Joining `citations` with both gives each citation the state of the citing patent and the state of the cited patent. These are left joins, so a citation is kept even when the cited patent is not in the patent file (mostly patents granted before 1963), and its state is NULL. The rows for patent 5654603 match the README example.

In [9]:
citing_state = patents.select(col("PATENT").alias("CITING"), col("POSTATE").alias("Citing_State"))
cited_state  = patents.select(col("PATENT").alias("CITED"),  col("POSTATE").alias("Cited_State"))

joined = (citations
          .join(citing_state, on="CITING", how="left")
          .join(cited_state,  on="CITED",  how="left")
          .select("CITED", "Cited_State", "CITING", "Citing_State"))

joined.filter(col("CITING") == 5654603).orderBy("CITED").show(10)

+-------+-----------+-------+------------+
|  CITED|Cited_State| CITING|Citing_State|
+-------+-----------+-------+------------+
| 872324|       NULL|5654603|          OH|
| 917138|       NULL|5654603|          OH|
|1231588|       NULL|5654603|          OH|
|1605112|       NULL|5654603|          OH|
|2134795|       NULL|5654603|          OH|
|2201699|       NULL|5654603|          OH|
|3031593|       NULL|5654603|          OH|
|3093764|         OH|5654603|          OH|
|3437858|         OH|5654603|          OH|
|3852137|         PA|5654603|          OH|
+-------+-----------+-------+------------+
only showing top 10 rows



### Step 3: Keep the same-state citations and count them
A citation counts only if the citing patent has a state and the cited patent has the same state, so rows with a NULL state on either side drop out. Grouping the remaining rows by CITING and counting gives each patent's number of same-state citations (`same_counts`).

In [10]:
same_state = joined.filter(col("Citing_State").isNotNull() & (col("Citing_State") == col("Cited_State")))
same_counts = same_state.groupBy("CITING").agg(count("*").alias("SAME_STATE"))
same_counts.orderBy(col("SAME_STATE").desc()).show(10)

+-------+----------+
| CITING|SAME_STATE|
+-------+----------+
|5959466|       125|
|5983822|       103|
|6008204|       100|
|5952345|        98|
|5998655|        96|
|5958954|        96|
|5936426|        94|
|5951547|        90|
|5978329|        90|
|5739256|        90|
+-------+----------+
only showing top 10 rows



### Step 4: Cache the counts and look at ties
`same_counts` is small but expensive to compute because it runs the joins over 16.5 million citations, so it is cached. 571,919 patents have at least one same-state citation. Six patents tie at 90 (5739256, 5913855, 5925042, 5951547, 5978329, 5980517), but only three fit in the top 10.

In [11]:
same_counts.cache()
print("patents with at least one same-state citation:", same_counts.count())
same_counts.filter(col("SAME_STATE") == 90).orderBy("CITING").show()

patents with at least one same-state citation: 571919
+-------+----------+
| CITING|SAME_STATE|
+-------+----------+
|5739256|        90|
|5913855|        90|
|5925042|        90|
|5951547|        90|
|5978329|        90|
|5980517|        90|
+-------+----------+



### Step 5: Add SAME_STATE to the patent table
A left join adds the SAME_STATE column to every patent, and patents with no same-state citation get NULL. The table is sorted by SAME_STATE descending (NULLs come last), with PATENT as a second key so tied rows always come out in the same order. This shows the top 10.

In [12]:
result = (patents
          .join(same_counts.withColumnRenamed("CITING", "PATENT"), on="PATENT", how="left")
          .orderBy(col("SAME_STATE").desc(), col("PATENT")))

result.show(10)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|SAME_STATE|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|5959466| 1999|14515|   1997|     US|     CA|    5310|      2|  NULL|   326|  4|    46|  159|       0|     1.0|   NULL|  0.6186|    NULL|  4.8868|  0.0455|   0.044|    NULL|    NULL|       125|
|5983822| 1999|14564|   1998|     US|     TX|  569900|      2|  NULL|   114|  5|    55|  200|       0|   0.995|   NULL|  0.7201|    NULL|   12.45|     0.0|     0.0|    NULL|    NULL|       103|
|6008204| 1999|14606|   1998| 

### Result
The top 10 counts are 125, 103, 100, 98, 96, 96, 94, 90, 90, 90, the same as the reference output. The only difference is the order of ties: here tied patents are ordered by patent number, and since six patents have 90, the reference image shows a different three of them.